# ARCADE Stenosis — DeepLabV3+ (ResNet101 encoder), Supervisor-Style Reporting

## Why this notebook

Your supervisor asked for two things:

1. **Model**: DeepLabV3+ with a ResNet101 encoder (his reference result: **Dice 0.65+, F1 0.55+**).
2. **Reporting**, in two blocks:
   - **Detection (BBOX)**: mAP@0.5, mAP@0.5:0.95, F1
   - **Instance Segmentation**: mAP@0.5, mAP@0.5:0.95, F1, Dice, IoU

Your work is segmentation (U-Net family), so the *Instance Segmentation* block is the main one. The *Detection (BBOX)*
block is still filled in, for two reasons: your YOLOv8 baseline reports exactly those numbers, and this notebook
derives boxes from the predicted lesions too, so the two models sit in the same table.

## Where the earlier models stood (test set)

| Model | Dice (mean) | IoU (mean) | F1 (lesion-level, IoU>=0.1) |
|---|---|---|---|
| U-Net + Attention U-Net ensemble (v2, best) | 0.5632 | 0.4225 | 0.7009 |
| Supervisor's reference | 0.65+ | - | 0.55+ |

Dice is the gap: 0.56 vs 0.65. That is what a stronger encoder is supposed to fix.

## What is different here (and why it should help)

- **Pretrained ResNet101 encoder (ImageNet)**. The U-Nets trained from scratch on 1000 images. A pretrained
  encoder already knows edges, tubes and textures, which is the usual biggest single gain on a small medical dataset.
  Images are normalised with ImageNet mean/std inside the model, so pretrained weights see the input they expect
  (and, unlike the YOLO experiment, the input here is a normal-looking X-ray, not a stacked filter composite).
- **DeepLabV3+** = atrous (dilated) pooling for multi-scale context + a decoder that brings back fine edges.
  Lesions here are small, so the decoder's low-level skip matters.
- **Two learning rates**: 1e-4 for the pretrained encoder, 1e-3 for the new decoder (standard fine-tuning practice).
- **Probability threshold chosen on the VALIDATION set**, not fixed at 0.5 and never tuned on test.
- Flip test-time augmentation and the small-blob filter from v2 are kept (both helped precision there).

## How the instance numbers are produced (important for honesty)

DeepLabV3+ is a *semantic* segmentation model: it outputs one mask per image. To get instance-level mAP and F1,
each connected blob in the predicted mask is treated as one predicted lesion, scored by its mean probability, with
its bounding box taken from the blob. Ground-truth instances come from the original annotation polygons.
Two lesions that touch will be merged into one blob, which slightly hurts instance metrics. Detection (BBOX) numbers
here are therefore *boxes derived from segmentation*, not from a detector.

## Read this before comparing with the supervisor's 55+ F1

"F1" depends on how a prediction is matched to a lesion. This notebook prints several, all clearly named:
`F1_mask@0.5` (mask IoU >= 0.5, the strict standard), `F1_mask@0.1`, and `F1_cc@0.1` (the exact definition used for
the v1/v2 U-Net numbers, so those stay comparable). Ask your supervisor which IoU threshold his 55+ used. The 0.70
from v2 was at the lenient 0.1 threshold, so it should not be compared with a 0.5-threshold number.

**No result is guaranteed.** The notebook prints whatever comes out, including whether the 0.65 Dice target was met.


## 1. Install & Import

In [ ]:
# ============================================================
# 1a. Installs
# ============================================================
!pip install -q segmentation-models-pytorch pycocotools scikit-image kagglehub albumentations
print("Installs complete.")


In [ ]:
# ============================================================
# 1b. Imports
# ============================================================
import os
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
import segmentation_models_pytorch as smp

from skimage.measure import label as cc_label, regionprops

print("Torch:", torch.__version__, "| smp:", smp.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2. Dataset

In [ ]:
# ============================================================
# 2a. Download ARCADE via kagglehub (explicitly the *stenosis* sub-task folder)
# ============================================================
import kagglehub

download_path = Path(kagglehub.dataset_download("nirmalgaud/arcade-dataset"))
candidates = list(download_path.rglob("train.json"))
if not candidates:
    raise FileNotFoundError(f"Could not find train.json under {download_path}.")

# ARCADE has several sub-tasks; only the 'stenosis' folder has category 26 = stenosis.
stenosis_candidates = [c for c in candidates if "stenosis" in str(c).lower()]
if not stenosis_candidates:
    raise FileNotFoundError(f"No 'stenosis' train.json found. Candidates: {[str(c) for c in candidates]}")
DATASET_ROOT = stenosis_candidates[0].parent.parent.parent
print("Resolved DATASET_ROOT:", DATASET_ROOT)
assert "stenosis" in str(DATASET_ROOT).lower()
for split in ["train", "val", "test"]:
    assert (DATASET_ROOT / split / "annotations" / f"{split}.json").exists()
    assert (DATASET_ROOT / split / "images").exists()
print("Dataset structure verified.")


In [ ]:
# ============================================================
# 2b. Google Drive (so weights and tables survive the session)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/arcade_stenosis_runs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive output dir:", DRIVE_OUTPUT_DIR)


## 3. Configuration

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================
WORK_ROOT = Path("/content/stenosis_dl3")
MASK_ROOT = WORK_ROOT / "masks"
SPLITS = ["train", "val", "test"]
STENOSIS_CATEGORY_ID = 26

# --- model (supervisor's choice) ---
ENCODER_NAME = "resnet101"
ENCODER_WEIGHTS = "imagenet"

# --- data / training ---
IMG_SIZE = 512            # native ARCADE resolution
BATCH_SIZE = 6            # if you get a CUDA out-of-memory error in Section 6, set this to 4
EPOCHS = 80
PATIENCE = 20
LR_ENCODER = 1e-4         # pretrained part: small steps
LR_DECODER = 1e-3         # new part: larger steps
DRIVE_BACKUP_EVERY = 10   # copy best.pt to Drive every N epochs (Colab sessions can die)

# --- loss ---
POS_WEIGHT = 3.0          # BCE weight on lesion pixels (v2 used 5 for the from-scratch U-Nets)
TVERSKY_ALPHA = 0.5       # 0.5/0.5 = plain Dice. Raise alpha (e.g. 0.6) if precision is low, lower it if recall is low.
TVERSKY_BETA = 0.5

# --- inference ---
THR_GRID = np.round(np.arange(0.20, 0.81, 0.05), 2)   # candidate probability thresholds, picked on VAL
DICE_TARGET, F1_TARGET = 0.65, 0.55                   # supervisor's reference numbers

# --- resume option: set SKIP_TRAINING=True and point CKPT_TO_EVALUATE at a saved best.pt to only run evaluation ---
SKIP_TRAINING = False
CKPT_TO_EVALUATE = "/content/drive/MyDrive/arcade_stenosis_runs/dl3_resnet101/best.pt"

for p in [WORK_ROOT, MASK_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
print("Config set. Device:", DEVICE)


## 4. Masks from Polygon Annotations (+ fail-fast checks)

In [ ]:
# ============================================================
# 4a. Rasterize polygons -> binary masks, resize image+mask to IMG_SIZE
# ============================================================
def load_coco(split):
    with open(DATASET_ROOT / split / "annotations" / f"{split}.json") as f:
        return json.load(f)

coco_data = {split: load_coco(split) for split in SPLITS}

def polygon_to_mask(segmentation, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    for poly in segmentation:
        pts = np.round(np.array(poly, dtype=np.float32).reshape(-1, 2)).astype(np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(mask, [pts], 1)
    return mask

def build_masks(split):
    coco = coco_data[split]
    img_dir = DATASET_ROOT / split / "images"
    out_img_dir = MASK_ROOT / split / "images"
    out_mask_dir = MASK_ROOT / split / "masks"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        if ann["category_id"] == STENOSIS_CATEGORY_ID:
            anns_by_image[ann["image_id"]].append(ann)

    for im in tqdm(coco["images"], desc=f"Building masks ({split})"):
        img = cv2.imread(str(img_dir / im["file_name"]), cv2.IMREAD_COLOR)
        h, w = im["height"], im["width"]
        mask = np.zeros((h, w), dtype=np.uint8)
        for ann in anns_by_image.get(im["id"], []):
            seg = ann.get("segmentation", [])
            if seg:
                mask |= polygon_to_mask(seg, h, w)
        img_r = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        mask_r = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        stem = Path(im["file_name"]).stem
        cv2.imwrite(str(out_img_dir / f"{stem}.png"), img_r)
        cv2.imwrite(str(out_mask_dir / f"{stem}.png"), mask_r * 255)

for split in SPLITS:
    build_masks(split)

# fail fast if masks are empty (wrong dataset folder = silent garbage, this happened once before)
_files = os.listdir(MASK_ROOT / "train" / "masks")
_nonempty = sum(1 for f in _files if cv2.imread(str(MASK_ROOT / "train" / "masks" / f), cv2.IMREAD_GRAYSCALE).max() > 0)
print(f"Sanity check: {_nonempty}/{len(_files)} train masks contain a lesion ({100*_nonempty/len(_files):.1f}%).")
assert 100 * _nonempty / len(_files) > 80, "Masks are (almost) empty -> DATASET_ROOT is the wrong sub-task folder."


In [ ]:
# ============================================================
# 4b. Data-driven minimum blob size for the false-positive filter (5th percentile of real lesion areas, train)
# ============================================================
train_lesion_areas = []
for f in os.listdir(MASK_ROOT / "train" / "masks"):
    m = cv2.imread(str(MASK_ROOT / "train" / "masks" / f), cv2.IMREAD_GRAYSCALE)
    for r in regionprops(cc_label((m > 127).astype(np.uint8))):
        train_lesion_areas.append(r.area)
train_lesion_areas = np.array(train_lesion_areas)
MIN_COMPONENT_AREA = int(np.percentile(train_lesion_areas, 5))
print(f"Lesion pixel areas (train): min={train_lesion_areas.min()}, 5th pct={MIN_COMPONENT_AREA}, "
      f"median={np.median(train_lesion_areas):.0f}")
print(f"MIN_COMPONENT_AREA = {MIN_COMPONENT_AREA}px (predicted blobs smaller than this are dropped).")


## 5. Metric Code

Everything that turns probability maps into the supervisor's numbers (instance extraction, COCO mAP for boxes and
masks, F1 at several IoU thresholds, Dice/IoU, validation threshold search). This block has no deep-learning
dependency and was unit-tested on synthetic cases with known answers (perfect prediction -> every metric 1.0,
shifted prediction -> 0, one extra false blob per image -> precision exactly 4/7, tiny blob removed by the filter).


In [ ]:
# ============================================================
# Metric code (unit-tested on synthetic cases with known answers before use)
# ============================================================
import io, contextlib, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
from skimage.measure import label as cc_label, regionprops
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval



# ------------------------------------------------------------------
# Ground truth (instance level) built from the ORIGINAL polygon annotations, scaled to IMG_SIZE
# ------------------------------------------------------------------
def build_gt_coco(coco_split, category_id=STENOSIS_CATEGORY_ID, img_size=IMG_SIZE):
    images, anns, gt_inst, fname_to_id = [], [], defaultdict(list), {}
    img_by_id = {im["id"]: im for im in coco_split["images"]}
    for im in coco_split["images"]:
        fname = f"{Path(im['file_name']).stem}.png"
        fname_to_id[fname] = im["id"]
        images.append({"id": im["id"], "file_name": fname, "width": img_size, "height": img_size})

    ann_id = 1
    for ann in coco_split["annotations"]:
        if ann["category_id"] != category_id:
            continue
        im = img_by_id[ann["image_id"]]
        sx, sy = img_size / im["width"], img_size / im["height"]
        polys = []
        for poly in ann.get("segmentation", []):
            if isinstance(poly, list) and len(poly) >= 6:
                arr = np.array(poly, dtype=np.float64).reshape(-1, 2)
                arr[:, 0] *= sx
                arr[:, 1] *= sy
                polys.append(arr.reshape(-1).tolist())
        if not polys:
            continue
        rle = maskUtils.merge(maskUtils.frPyObjects(polys, img_size, img_size))
        area = float(maskUtils.area(rle))
        if area <= 0:
            continue
        bbox = [float(v) for v in maskUtils.toBbox(rle)]
        anns.append({"id": ann_id, "image_id": ann["image_id"], "category_id": 1,
                     "segmentation": polys, "bbox": bbox, "area": area, "iscrowd": 0})
        gt_inst[ann["image_id"]].append({"rle": rle, "bbox": bbox})
        ann_id += 1

    gt_dict = {"images": images, "annotations": anns, "categories": [{"id": 1, "name": "stenosis"}]}
    return gt_dict, gt_inst, fname_to_id


# ------------------------------------------------------------------
# Binarize / filter / instances
# ------------------------------------------------------------------
def binarize_and_filter(prob, thr, min_area):
    mask = (prob.astype(np.float32) > thr).astype(np.uint8)
    if min_area <= 0:
        return mask
    cc = cc_label(mask)
    out = np.zeros_like(mask)
    for r in regionprops(cc):
        if r.area >= min_area:
            out[cc == r.label] = 1
    return out


def extract_instances_from_mask(mask, prob):
    cc = cc_label(mask)
    prob32 = prob.astype(np.float32)
    insts = []
    for r in regionprops(cc):
        comp = (cc == r.label)
        rle = maskUtils.encode(np.asfortranarray(comp.astype(np.uint8)))
        y0, x0, y1, x1 = r.bbox
        insts.append({"rle": rle, "bbox": [float(x0), float(y0), float(x1 - x0), float(y1 - y0)],
                      "score": float(prob32[comp].mean())})
    return insts


# ------------------------------------------------------------------
# Pixel metrics
# ------------------------------------------------------------------
def mask_dice_iou(pred, gt, eps=1e-6):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = float(np.logical_and(pred, gt).sum())
    p, g = float(pred.sum()), float(gt.sum())
    dice = (2 * inter + eps) / (p + g + eps)
    iou = (inter + eps) / (p + g - inter + eps)
    return dice, iou, inter, p, g


# ------------------------------------------------------------------
# Instance-level F1 (greedy one-to-one matching, highest score first)
# ------------------------------------------------------------------
def instance_f1(per_image_preds, gt_inst, image_ids, iou_thr, use="mask"):
    TP = FP = FN = 0
    for iid in image_ids:
        preds = sorted(per_image_preds.get(iid, []), key=lambda d: -d["score"])
        gts = gt_inst.get(iid, [])
        if not preds:
            FN += len(gts)
            continue
        if not gts:
            FP += len(preds)
            continue
        if use == "mask":
            ious = maskUtils.iou([p["rle"] for p in preds], [g["rle"] for g in gts], [0] * len(gts))
        else:
            ious = maskUtils.iou([p["bbox"] for p in preds], [g["bbox"] for g in gts], [0] * len(gts))
        ious = np.array(ious).reshape(len(preds), len(gts))
        matched = set()
        for i in range(len(preds)):
            best_j, best_iou = -1, 0.0
            for j in range(len(gts)):
                if j in matched:
                    continue
                if ious[i, j] > best_iou:
                    best_iou, best_j = ious[i, j], j
            if best_j >= 0 and best_iou >= iou_thr:
                TP += 1
                matched.add(best_j)
            else:
                FP += 1
        FN += len(gts) - len(matched)
    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return {"P": precision, "R": recall, "F1": f1, "TP": TP, "FP": FP, "FN": FN}


# ------------------------------------------------------------------
# COCO mAP (bbox or segm)
# ------------------------------------------------------------------
def coco_map(gt_dict, per_image_preds, use="segm"):
    dts = []
    for iid, preds in per_image_preds.items():
        for p in preds:
            if use == "bbox":
                dts.append({"image_id": iid, "category_id": 1, "bbox": p["bbox"], "score": p["score"]})
            else:
                counts = p["rle"]["counts"]
                counts = counts.decode("ascii") if isinstance(counts, bytes) else counts
                dts.append({"image_id": iid, "category_id": 1,
                            "segmentation": {"size": p["rle"]["size"], "counts": counts}, "score": p["score"]})
    if not dts:
        return 0.0, 0.0
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        gt = COCO()
        gt.dataset = copy.deepcopy(gt_dict)
        gt.createIndex()
        dt = gt.loadRes(dts)
        ev = COCOeval(gt, dt, "bbox" if use == "bbox" else "segm")
        ev.evaluate(); ev.accumulate(); ev.summarize()
    return float(ev.stats[1]), float(ev.stats[0])      # mAP@0.5, mAP@0.5:0.95


# ------------------------------------------------------------------
# Old (v1/v2) connected-component lesion matching, kept so numbers stay comparable
# ------------------------------------------------------------------
def lesion_level_match(pred_mask, gt_mask, iou_thres=0.10):
    pred_cc, gt_cc = cc_label(pred_mask), cc_label(gt_mask)
    pred_regions, gt_regions = regionprops(pred_cc), regionprops(gt_cc)
    matched_gt, TP, FP = set(), 0, 0
    for pr in pred_regions:
        pr_mask = (pred_cc == pr.label)
        best_iou, best_idx = 0.0, -1
        for gi, gr in enumerate(gt_regions):
            if gi in matched_gt:
                continue
            gr_mask = (gt_cc == gr.label)
            inter = np.logical_and(pr_mask, gr_mask).sum()
            union = np.logical_or(pr_mask, gr_mask).sum()
            iou = inter / union if union > 0 else 0.0
            if iou > best_iou:
                best_iou, best_idx = iou, gi
        if best_iou >= iou_thres:
            TP += 1
            matched_gt.add(best_idx)
        else:
            FP += 1
    return TP, FP, len(gt_regions) - len(matched_gt)


# ------------------------------------------------------------------
# Full evaluation of a dict of probability maps
# ------------------------------------------------------------------
def evaluate_maps(prob_maps, gt_masks, gt_dict, gt_inst, fname_to_id, thr, min_area):
    per_image_preds = {}
    dice_l, iou_l = [], []
    inter_s = p_s = g_s = 0.0
    cTP = cFP = cFN = 0
    for fname, prob in prob_maps.items():
        gt = gt_masks[fname]
        pm = binarize_and_filter(prob, thr, min_area)
        d, i, inter, p, g = mask_dice_iou(pm, gt)
        dice_l.append(d); iou_l.append(i)
        inter_s += inter; p_s += p; g_s += g
        per_image_preds[fname_to_id[fname]] = extract_instances_from_mask(pm, prob)
        tp, fp, fn = lesion_level_match(pm, gt.astype(np.uint8))
        cTP += tp; cFP += fp; cFN += fn

    image_ids = [fname_to_id[f] for f in prob_maps]
    seg50, seg5095 = coco_map(gt_dict, per_image_preds, "segm")
    box50, box5095 = coco_map(gt_dict, per_image_preds, "bbox")
    f_mask50 = instance_f1(per_image_preds, gt_inst, image_ids, 0.5, "mask")
    f_mask10 = instance_f1(per_image_preds, gt_inst, image_ids, 0.1, "mask")
    f_box50 = instance_f1(per_image_preds, gt_inst, image_ids, 0.5, "bbox")
    cP = cTP / (cTP + cFP + 1e-9); cR = cTP / (cTP + cFN + 1e-9)
    return {
        "Dice_mean": float(np.mean(dice_l)), "IoU_mean": float(np.mean(iou_l)),
        "Dice_global": 2 * inter_s / (p_s + g_s + 1e-9), "IoU_global": inter_s / (p_s + g_s - inter_s + 1e-9),
        "seg_mAP50": seg50, "seg_mAP50_95": seg5095, "box_mAP50": box50, "box_mAP50_95": box5095,
        "F1_mask@0.5": f_mask50["F1"], "P_mask@0.5": f_mask50["P"], "R_mask@0.5": f_mask50["R"],
        "F1_mask@0.1": f_mask10["F1"], "F1_bbox@0.5": f_box50["F1"],
        "F1_cc@0.1(v1/v2 style)": 2 * cP * cR / (cP + cR + 1e-9), "P_cc@0.1": cP, "R_cc@0.1": cR,
    }


def choose_threshold(val_prob_maps, val_gt_masks, min_area, grid):
    table = []
    for thr in grid:
        ds = [mask_dice_iou(binarize_and_filter(prob, thr, min_area), val_gt_masks[f])[0]
              for f, prob in val_prob_maps.items()]
        table.append((float(thr), float(np.mean(ds))))
    best = max(table, key=lambda t: t[1])
    return best[0], table


In [ ]:
# ============================================================
# 5b. Ground truth for VAL (masks only) and TEST (masks + instance annotations)
# ============================================================
def load_gt_masks(split):
    d = {}
    for f in sorted(os.listdir(MASK_ROOT / split / "masks")):
        d[f] = cv2.imread(str(MASK_ROOT / split / "masks" / f), cv2.IMREAD_GRAYSCALE) > 127
    return d

val_gt_masks = load_gt_masks("val")
test_gt_masks = load_gt_masks("test")

test_gt_dict, test_gt_inst, test_fname_to_id = build_gt_coco(coco_data["test"], STENOSIS_CATEGORY_ID, IMG_SIZE)
n_gt_inst = sum(len(v) for v in test_gt_inst.values())
print(f"Test: {len(test_gt_masks)} images, {n_gt_inst} ground-truth lesion instances")
assert n_gt_inst > 0
assert set(test_gt_masks) == set(test_fname_to_id), "mask files and COCO image names do not line up"


## 6. Data Loading, Model, Loss

In [ ]:
# ============================================================
# 6a. Dataset + augmentation (same recipe as the U-Net notebooks)
# ============================================================
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(rotate=(-10, 10), translate_percent=(0.0, 0.10), scale=(0.85, 1.15), shear=(-2, 2), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(p=0.2),
])

class StenosisSegDataset(Dataset):
    def __init__(self, split, transform=None):
        self.img_dir = MASK_ROOT / split / "images"
        self.mask_dir = MASK_ROOT / split / "masks"
        self.files = sorted(os.listdir(self.img_dir))
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        fname = self.files[idx]
        img = cv2.cvtColor(cv2.imread(str(self.img_dir / fname)), cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(str(self.mask_dir / fname), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        if self.transform is not None:
            out = self.transform(image=img, mask=mask)
            img, mask = out["image"], out["mask"]
        img = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1)   # [0,1]; model normalises itself
        return img, torch.from_numpy(mask).unsqueeze(0), fname

train_ds = StenosisSegDataset("train", transform=train_transform)
val_ds = StenosisSegDataset("val")
test_ds = StenosisSegDataset("test")
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


In [ ]:
# ============================================================
# 6b. Model: DeepLabV3+ with a ResNet101 encoder. Inputs are [0,1] images; ImageNet normalisation is built in.
# ============================================================
class NormalizedSegModel(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.net((x - self.mean) / self.std)      # logits, shape (B, 1, H, W)

def build_model(encoder_weights=ENCODER_WEIGHTS):
    net = smp.DeepLabV3Plus(encoder_name=ENCODER_NAME, encoder_weights=encoder_weights, in_channels=3, classes=1)
    return NormalizedSegModel(net)

# ============================================================
# 6c. Loss: weighted BCE + Tversky (Tversky with alpha=beta=0.5 is Dice)
# ============================================================
def tversky_loss(logits, targets, alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA, eps=1e-6):
    probs = torch.sigmoid(logits).view(logits.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    TP = (probs * targets).sum(dim=1)
    FP = (probs * (1 - targets)).sum(dim=1)
    FN = ((1 - probs) * targets).sum(dim=1)
    return 1 - ((TP + eps) / (TP + alpha * FP + beta * FN + eps)).mean()

class SegLoss(nn.Module):
    def __init__(self, pos_weight=POS_WEIGHT, bce_wt=0.5, tversky_wt=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_weight)))
        self.bce_wt, self.tversky_wt = bce_wt, tversky_wt
    def forward(self, logits, targets):
        return self.bce_wt * self.bce(logits, targets) + self.tversky_wt * tversky_loss(logits, targets)

@torch.no_grad()
def batch_dice(logits, targets, thr=0.5, eps=1e-6):
    preds = (torch.sigmoid(logits) > thr).float().view(logits.size(0), -1)
    t = targets.view(targets.size(0), -1)
    inter = (preds * t).sum(dim=1)
    return ((2 * inter + eps) / (preds.sum(dim=1) + t.sum(dim=1) + eps)).mean().item()


In [ ]:
# ============================================================
# 6d. SMOKE TEST (about 20 seconds): proves the model builds, runs forward+backward at the real batch size,
#     and tells you the GPU memory use BEFORE a long training run starts.
# ============================================================
_m = build_model().to(DEVICE)
_x = torch.rand(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
_y = (torch.rand(BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE, device=DEVICE) > 0.99).float()
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
try:
    with autocast("cuda", enabled=(DEVICE.type == "cuda")):
        _out = _m(_x)
        _loss = SegLoss().to(DEVICE)(_out.float(), _y)
    _loss.backward()
except torch.cuda.OutOfMemoryError:
    raise RuntimeError(f"CUDA out of memory at BATCH_SIZE={BATCH_SIZE}. Set BATCH_SIZE = 4 in Section 3 and re-run from there.")
assert _out.shape == (BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE), _out.shape
assert torch.isfinite(_loss)
n_params = sum(p.numel() for p in _m.parameters()) / 1e6
print(f"Smoke test OK: output {tuple(_out.shape)}, loss {_loss.item():.4f}, params {n_params:.1f}M")
if DEVICE.type == "cuda":
    print(f"Peak GPU memory for one training step: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
del _m, _x, _y, _out, _loss
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 7. Training

In [ ]:
# ============================================================
# 7a. Training function (two learning rates, cosine schedule, mixed precision, early stopping on val Dice)
# ============================================================
def train_model():
    model = build_model().to(DEVICE)
    criterion = SegLoss().to(DEVICE)

    enc_params = list(model.net.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
    optimizer = torch.optim.AdamW(
        [{"params": enc_params, "lr": LR_ENCODER}, {"params": dec_params, "lr": LR_DECODER}], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    ckpt_dir = WORK_ROOT / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path = ckpt_dir / "best.pt"
    drive_dir = DRIVE_OUTPUT_DIR / "dl3_resnet101"
    drive_dir.mkdir(parents=True, exist_ok=True)

    history, best_val, no_improve = [], -1.0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(imgs)
            loss = criterion(logits.float(), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * imgs.size(0)
        train_loss = loss_sum / len(train_ds)
        scheduler.step()

        model.eval()
        d_sum, n = 0.0, 0
        with torch.no_grad():
            for imgs, masks, _ in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                d_sum += batch_dice(model(imgs).float(), masks) * imgs.size(0)
                n += imgs.size(0)
        val_dice = d_sum / n
        history.append({"epoch": epoch, "train_loss": train_loss, "val_dice": val_dice})
        print(f"epoch {epoch:3d}/{EPOCHS} | train_loss {train_loss:.4f} | val_dice {val_dice:.4f} | best {max(best_val, val_dice):.4f}")

        if val_dice > best_val:
            best_val, no_improve = val_dice, 0
            torch.save(model.state_dict(), best_path)
        else:
            no_improve += 1
        if epoch % DRIVE_BACKUP_EVERY == 0 and best_path.exists():
            shutil.copy(best_path, drive_dir / "best.pt")
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val_dice = {best_val:.4f}")
            break

    shutil.copy(best_path, drive_dir / "best.pt")
    return pd.DataFrame(history), best_path, best_val


In [ ]:
# ============================================================
# 7b. Run training (or just point at a saved checkpoint)
# ============================================================
if SKIP_TRAINING:
    best_path = Path(CKPT_TO_EVALUATE)
    history_df = None
    print("SKIP_TRAINING=True -> using checkpoint:", best_path)
else:
    history_df, best_path, best_val = train_model()
    print(f"Best val Dice: {best_val:.4f} | checkpoint: {best_path}")


In [ ]:
# ============================================================
# 7c. Training curves
# ============================================================
if history_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"]); axes[0].set_title("Train loss")
    axes[1].plot(history_df["epoch"], history_df["val_dice"]); axes[1].set_title("Val Dice (threshold 0.5)")
    for ax in axes:
        ax.set_xlabel("epoch")
    plt.tight_layout(); plt.show()


## 8. Inference: Probability Maps, Threshold Chosen on Validation

Probability maps are computed once per image and cached, so every variant below is just a different way of
cutting the same maps. The threshold is picked by mean Dice on the **validation** images (with the blob filter
applied) and then used unchanged on the test set.


In [ ]:
# ============================================================
# 8a. Load best weights, build cached probability maps
# ============================================================
model = build_model(encoder_weights=None).to(DEVICE)       # weights come from the checkpoint, no download needed
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

def load_image_tensor(split, fname):
    img = cv2.cvtColor(cv2.imread(str(MASK_ROOT / split / "images" / fname)), cv2.COLOR_BGR2RGB)
    return torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

@torch.no_grad()
def get_prob_maps(split, tta):
    out = {}
    for fname in tqdm(sorted(os.listdir(MASK_ROOT / split / "images")), desc=f"{split} probs (TTA={tta})"):
        x = load_image_tensor(split, fname)
        p = torch.sigmoid(model(x).float())[0, 0]
        if tta:
            pf = torch.sigmoid(model(torch.flip(x, dims=[3])).float())[0, 0]
            p = (p + torch.flip(pf, dims=[1])) / 2.0
        out[fname] = p.cpu().numpy().astype(np.float16)
    return out

val_probs_tta = get_prob_maps("val", tta=True)
test_probs_base = get_prob_maps("test", tta=False)
test_probs_tta = get_prob_maps("test", tta=True)


In [ ]:
# ============================================================
# 8b. Pick the probability threshold on VAL (mean Dice, blob filter on)
# ============================================================
best_thr, sweep = choose_threshold(val_probs_tta, val_gt_masks, MIN_COMPONENT_AREA, THR_GRID)
print("threshold -> val mean Dice")
for t, d in sweep:
    print(f"  {t:.2f} -> {d:.4f}" + ("   <-- chosen" if t == best_thr else ""))
print(f"\nChosen threshold (from validation only): {best_thr}")


## 9. Test-Set Evaluation (four variants so each step's effect is visible)

| Variant | What it is |
|---|---|
| A | single pass, threshold 0.5, no filter |
| B | + flip test-time augmentation |
| C | + small-blob filter |
| **D (final)** | + threshold chosen on validation. This is the row to report; A-C are the ablation. |


In [ ]:
# ============================================================
# 9. Evaluate variants A-D
# ============================================================
variants = {
    "A base":            (test_probs_base, 0.5, 0),
    "B +TTA":            (test_probs_tta, 0.5, 0),
    "C +TTA+filter":     (test_probs_tta, 0.5, MIN_COMPONENT_AREA),
    "D +TTA+filter+thr": (test_probs_tta, best_thr, MIN_COMPONENT_AREA),
}
results = {}
for name, (probs, thr, min_area) in variants.items():
    results[name] = evaluate_maps(probs, test_gt_masks, test_gt_dict, test_gt_inst, test_fname_to_id, thr, min_area)
    print(f"{name:20s} done  (Dice_mean={results[name]['Dice_mean']:.4f})")


## 10. Supervisor-Style Tables

In [ ]:
# ============================================================
# 10. Tables (narrow on purpose, so nothing is cut off in the printout / PDF export)
# ============================================================
r = lambda v: round(float(v), 4)

seg_table = pd.DataFrame([{
    "Method": f"DeepLabV3+ R101 [{k}]",
    "mAP@0.5": r(v["seg_mAP50"]), "mAP@.5:.95": r(v["seg_mAP50_95"]), "F1(IoU.5)": r(v["F1_mask@0.5"]),
    "Dice": r(v["Dice_mean"]), "IoU": r(v["IoU_mean"])} for k, v in results.items()])

det_rows = [{
    "Method": f"DeepLabV3+ R101 [{k}] (boxes from masks)",
    "mAP@0.5": r(v["box_mAP50"]), "mAP@.5:.95": r(v["box_mAP50_95"]), "F1(IoU.5)": r(v["F1_bbox@0.5"])}
    for k, v in results.items()]
det_rows.append({"Method": "YOLOv8 raw (earlier notebook, real detector)", "mAP@0.5": 0.2541, "mAP@.5:.95": 0.0847, "F1(IoU.5)": 0.3292})
det_table = pd.DataFrame(det_rows)

other_table = pd.DataFrame([{
    "Method": k, "Dice_global": r(v["Dice_global"]), "F1_mask@.1": r(v["F1_mask@0.1"]),
    "F1_cc@.1(v1/v2)": r(v["F1_cc@0.1(v1/v2 style)"]), "P_mask@.5": r(v["P_mask@0.5"]), "R_mask@.5": r(v["R_mask@0.5"])}
    for k, v in results.items()])

print("=== INSTANCE SEGMENTATION ===")
print(seg_table.to_string(index=False))
print("\n=== DETECTION (BBOX) ===")
print(det_table.to_string(index=False))
print("\n=== EXTRA: other Dice / F1 definitions (so any comparison is apples-to-apples) ===")
print(other_table.to_string(index=False))
print("\nContext: U-Net+Attention-U-Net ensemble (v2): Dice 0.5632, IoU 0.4225, F1_cc@.1 0.7009")

seg_table.to_csv(WORK_ROOT / "instance_segmentation_table.csv", index=False)
det_table.to_csv(WORK_ROOT / "detection_bbox_table.csv", index=False)
other_table.to_csv(WORK_ROOT / "extra_metrics_table.csv", index=False)

final = results["D +TTA+filter+thr"]
print("\n=== TARGET CHECK (final variant D) ===")
print(f"Dice (mean per image) {final['Dice_mean']:.4f}  vs target {DICE_TARGET}: {'REACHED' if final['Dice_mean'] >= DICE_TARGET else 'not reached'}")
print(f"Dice (global)         {final['Dice_global']:.4f}  vs target {DICE_TARGET}: {'REACHED' if final['Dice_global'] >= DICE_TARGET else 'not reached'}")
for key in ["F1_mask@0.5", "F1_mask@0.1", "F1_cc@0.1(v1/v2 style)", "F1_bbox@0.5"]:
    print(f"{key:24s} {final[key]:.4f}  vs target {F1_TARGET}: {'REACHED' if final[key] >= F1_TARGET else 'not reached'}")


## 11. Look at the Predictions (final variant D)

In [ ]:
# ============================================================
# 11. Overlays: green = prediction, red = ground truth
# ============================================================
files = random.sample(sorted(test_gt_masks), 6)
fig, axes = plt.subplots(len(files), 2, figsize=(10, 5 * len(files)))
for i, fname in enumerate(files):
    img = cv2.cvtColor(cv2.imread(str(MASK_ROOT / "test" / "images" / fname)), cv2.COLOR_BGR2RGB)
    gt = test_gt_masks[fname]
    pm = binarize_and_filter(test_probs_tta[fname], best_thr, MIN_COMPONENT_AREA).astype(bool)

    o1 = img.copy(); o1[gt] = [255, 0, 0]
    axes[i, 0].imshow(cv2.addWeighted(img, 0.6, o1, 0.4, 0)); axes[i, 0].set_title(f"Ground truth: {fname}"); axes[i, 0].axis("off")
    o2 = img.copy(); o2[pm] = [0, 255, 0]
    d = mask_dice_iou(pm, gt)[0]
    axes[i, 1].imshow(cv2.addWeighted(img, 0.6, o2, 0.4, 0)); axes[i, 1].set_title(f"Prediction (Dice {d:.2f})"); axes[i, 1].axis("off")
plt.tight_layout(); plt.show()


## 12. Save to Google Drive

In [ ]:
out_dir = DRIVE_OUTPUT_DIR / "dl3_resnet101"
out_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(best_path, out_dir / "best.pt")
for name in ["instance_segmentation_table.csv", "detection_bbox_table.csv", "extra_metrics_table.csv"]:
    shutil.copy(WORK_ROOT / name, out_dir / name)
if history_df is not None:
    history_df.to_csv(out_dir / "training_history.csv", index=False)
with open(out_dir / "chosen_threshold.json", "w") as f:
    json.dump({"threshold": float(best_thr), "min_component_area": int(MIN_COMPONENT_AREA)}, f)
print("Saved to:", out_dir)


## Notes & Next Steps

- **Report row D** as the model's result (its threshold was chosen on validation, not on test). Rows A-C are the
  ablation showing what each step added. Report the Dice *and* say which definition (per-image mean vs global).
- **If Dice is still below 0.65**: first look at the validation-threshold table and at the Section 11 overlays.
  If predictions are the right place but too thin/thick, adjust `TVERSKY_ALPHA/BETA` (Section 3) and retrain.
  If many lesions are missed, lower `POS_WEIGHT` is *not* the answer; raise it. Averaging this model with the
  U-Net models from the earlier notebook (same idea as the v2 ensemble, which gave +0.02 F1) is the next
  cheap step, because different architectures make different mistakes.
- **If the session dies mid-training**: `best.pt` is copied to Drive every 10 epochs. Set `SKIP_TRAINING = True`
  and `CKPT_TO_EVALUATE` in Section 3, then run all cells to get the tables from the saved weights.
- Next thesis objectives (Grad-CAM explainability, cross-dataset test on DCA1/CADICA) both start from this
  checkpoint.
